# Task 4 — Transform to SILVER (SCD Type 1) — TRUE branch

Reads the **new** rows (same `window_minutes` window as validate), cleans them — trim, cast, dedupe, drop incomplete rows, compute `total_amount` — then **MERGEs them into `<catalog>.silver.sales_orders`**.

**SCD Type 1 in one sentence:** silver keeps **one row per `order_id`** — if the same order arrives again with new values, it is **updated in place** (latest wins); no history is kept. (SCD Type 2 would keep every version with history — not needed here.)

Only runs when the condition task said TRUE (`row_count > 0`). Gold KPIs are built in the next task, `gold_kpi` (notebook 06_gold_kpi).

In [0]:
# CELL 1 — Parameters (same catalog + window as the validate task)
dbutils.widgets.text("catalog", "demo1", "Catalog that holds all demo tables")
dbutils.widgets.text("window_minutes", "10", "Look-back window in minutes")
dbutils.widgets.text("job_id", "interactive", "Job ID (passed by the job)")
dbutils.widgets.text("run_id", "interactive", "Run ID (passed by the job)")

catalog = dbutils.widgets.get("catalog")
window_minutes = int(dbutils.widgets.get("window_minutes"))
job_id = dbutils.widgets.get("job_id")
run_id = dbutils.widgets.get("run_id")

BRONZE_TABLE = f"{catalog}.bronze.sales_orders"
SILVER_TABLE = f"{catalog}.silver.sales_orders"
print(f"catalog : {catalog} | window : last {window_minutes} minutes")
print(f"job_id / run_id: {job_id} / {run_id}")

In [0]:
# CELL 2 — Read the NEW rows (ingested in the last N minutes)
# Same timestamp rule as validate — so this task transforms exactly
# the rows that were counted: the window is the contract between tasks.
from pyspark.sql import functions as F

new_rows_df = spark.table(BRONZE_TABLE).where(
    F.col("ingested_at") >= F.current_timestamp() - F.expr(f"INTERVAL {window_minutes} MINUTES")
)
rows_before = new_rows_df.count()
print(f"New rows to transform: {rows_before}")

In [0]:
# CELL 3 — Clean & transform (bronze -> gold logic)
# 1. trim()  -> remove padded spaces
# 2. cast()  -> quantity to int, unit_price to decimal (missing -> NULL)
# 3. dropDuplicates by order_id
# 4. quality gate -> keep only complete, positive rows
# 5. total_amount = quantity * unit_price
clean_df = (
    new_rows_df
    .withColumn("customer_name", F.trim(F.col("customer_name")))
    .withColumn("category", F.trim(F.col("category")))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("decimal(10,2)"))
    .withColumn("order_date", F.to_date(F.col("order_date"), "yyyy-MM-dd"))
    .dropDuplicates(["order_id"])
    .where(
        F.col("order_id").isNotNull()
        & F.col("quantity").isNotNull()
        & F.col("unit_price").isNotNull()
        & (F.col("quantity") > 0)
        & (F.col("unit_price") > 0)
    )
    .withColumn("total_amount", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .select("order_id", "customer_name", "product", "category",
            "quantity", "unit_price", "total_amount", "order_date")
)

rows_after = clean_df.count()
print(f"Rows in (raw): {rows_before} | Rows out (clean): {rows_after} | dropped: {rows_before - rows_after}")
display(clean_df)

In [0]:
# CELL 4 — SCD Type 1 MERGE into silver + audit row
# WHAT IS HAPPENING:
#   * WHEN MATCHED (same order_id seen again) -> UPDATE in place (latest wins)
#   * WHEN NOT MATCHED (brand-new order)     -> INSERT
#   * Result: one row per order_id, always the latest values. That is SCD1.
from datetime import datetime

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
        order_id STRING, customer_name STRING, product STRING, category STRING,
        quantity INT, unit_price DECIMAL(10,2), total_amount DECIMAL(18,2),
        order_date DATE, updated_at TIMESTAMP
    )
""")

silver_df = (clean_df
    .withColumn("total_amount", F.col("total_amount").cast("decimal(18,2)"))
    .withColumn("updated_at", F.current_timestamp()))
silver_df.createOrReplaceTempView("silver_updates")

result = spark.sql(f"""
    MERGE INTO {SILVER_TABLE} AS t
    USING silver_updates AS s
    ON t.order_id = s.order_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""").collect()[0]

print(f"SCD1 MERGE: {result['num_inserted_rows']} orders inserted, "
      f"{result['num_updated_rows']} orders updated in place")

log_df = spark.createDataFrame(
    [(datetime.now(), job_id, run_id, "transform_publish", int(rows_after), "SUCCESS",
      f"merged to silver (raw={rows_before}, clean={rows_after})")],
    "logged_at TIMESTAMP, job_id STRING, run_id STRING, task_name STRING, "
    "rows_processed BIGINT, status STRING, message STRING",
)
log_df.write.mode("append").saveAsTable(f"{catalog}.bronze.job_run_log")

print(f"Silver now holds {spark.table(SILVER_TABLE).count()} rows (one per order_id):")
display(spark.table(SILVER_TABLE).orderBy(F.col("updated_at").desc()).limit(15))

## Task 4 — recap

* New rows (last N minutes) → cleaned → **SCD1 MERGE into `<catalog>.silver.sales_orders`**: one row per order_id, latest values win, no history.
* Show the MERGE metrics in the presentation — first run: "8 inserted, 0 updated"; run the same data again: "0 inserted, 8 updated". That IS SCD1 live.
* Gold KPIs are built in the next task, `gold_kpi` (notebook 06_gold_kpi).